<a href="https://colab.research.google.com/github/adriatek/waymo-scene-verifier/blob/main/Waymo_Explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install waymo-open-dataset-tf-2-12-0==1.6.7 --no-deps
!pip install protobuf==3.20.0

from waymo_open_dataset import dataset_pb2 as open_dataset
print("imports fine, we're in business")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-monitoring 2.31.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
google-cloud-functions 1.24.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 3.20.0 which is incompatible.
google-cloud-datastore 2.25.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21

imports fine, we're in business


In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
!gsutil ls gs://waymo_open_dataset_v_1_4_2/individual_files/training/ | head -5

gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord
gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10023947602400723454_1120_000_1140_000_with_camera_labels.tfrecord
gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-1005081002024129653_5313_150_5333_150_with_camera_labels.tfrecord
gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10061305430875486848_1080_000_1100_000_with_camera_labels.tfrecord
gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10072140764565668044_4060_000_4080_000_with_camera_labels.tfrecord
Exception ignored in: <_io.TextIOWrapper name='<stdout>' mode='w' encoding='utf-8'>
BrokenPipeError: [Errno 32] Broken pipe


In [4]:


def read_in_segments():
  segment_files = ['gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord','gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10023947602400723454_1120_000_1140_000_with_camera_labels.tfrecord', 'gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-1005081002024129653_5313_150_5333_150_with_camera_labels.tfrecord','gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10061305430875486848_1080_000_1100_000_with_camera_labels.tfrecord','gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10072140764565668044_4060_000_4080_000_with_camera_labels.tfrecord']

  for idx, file in enumerate(segment_files, start=1):
    !gsutil cp {file} /content/segment{idx}.tfrecord
    print(f"copied segment{idx}.tfrecord")



read_in_segments()

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

| [1 files][  1.0 GiB/  1.0 GiB]   71.2 MiB/s                                   
Operation completed over 1 objects/1.0 GiB.                                      
copied segment1.tfrecord
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration 

In [5]:
from waymo_open_dataset import label_pb2

print(label_pb2.Label.Type.items())

[('TYPE_UNKNOWN', 0), ('TYPE_VEHICLE', 1), ('TYPE_PEDESTRIAN', 2), ('TYPE_SIGN', 3), ('TYPE_CYCLIST', 4)]


In [6]:
!pip install "protobuf==5.29.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.0
    Uninstalling protobuf-3.20.0:
      Successfully uninstalled protobuf-3.20.0


In [7]:
# extract first frame metadata into object and return it
import os
import pandas as pd
import tensorflow as tf
from waymo_open_dataset import dataset_pb2 as open_dataset

def extract_segment_metadata(filepath):
  segment_metadata = {}
  dataset = tf.data.TFRecordDataset(filepath, compression_type='')

  frame = open_dataset.Frame()
  for data in dataset:
    frame.ParseFromString(bytearray(data.numpy()))
    segment_metadata["segment_id"] = frame.context.name
    segment_metadata["time_of_day"] = frame.context.stats.time_of_day
    segment_metadata["weather"] = frame.context.stats.weather
    segment_metadata["location"] = frame.context.stats.location

    segment_metadata["vehicle_count"] = 0
    segment_metadata["pedestrian_count"] = 0
    segment_metadata["cyclist_count"] = 0

    for item in frame.context.stats.laser_object_counts:
      if item.type == 0: # TYPE UNKNOWN
        pass
      elif item.type == 1: # TYPE VEHICLE
        segment_metadata["vehicle_count"] += item.count
      elif item.type == 2: # TYPE PEDESTRIAN
        segment_metadata["pedestrian_count"] += item.count
      elif item.type == 3: # TYPE SIGN
        pass
      elif item.type == 4: # TYPE CYCLIST
        segment_metadata["cyclist_count"] += item.count
      else:
        pass

    return segment_metadata



folder = '/content'
all_files = sorted(os.listdir(folder))

rows = []
#for each segment, extract metadata object and store in rows array
for segment_file in all_files:
  if segment_file.endswith(".tfrecord"):
    segment_file = os.path.join(folder, segment_file)
    segment_metadata = extract_segment_metadata(segment_file)
    rows.append(segment_metadata)



In [8]:
# Connect to the SQLite file in your Colab folder
import sqlite3

conn = sqlite3.connect('/content/waymo_segments.db')
segment_df = pd.DataFrame(rows)

segment_df.to_sql("segments", conn, if_exists="append", index=False)

# Now, write an SQL query on the table we made
query = "SELECT weather, COUNT(*) as total_segments FROM segments GROUP BY weather"
result = pd.read_sql_query(query, conn)

conn.close()

result.head()

# save as CSV
result.to_csv('/content/waymo_results.csv', index=False)
print("CSV file successfully saved in your Colab folder!")




CSV file successfully saved in your Colab folder!


In [19]:
def check_high_pedestrian_density(df):
  threshold = 6

  filtered_df = df[df["pedestrian_count"] > threshold]
  findings = []

  for idx, row in filtered_df.iterrows():
    findings.append({
        "check_id": f"C1",
        "segment_id": row["segment_id"],
        "severity": "warning",
        "description": f"{row["pedestrian_count"]} pedestrians exceeds threshold: {threshold}."
    })

  return findings

def check_vulnerable_road_user(df):
  filtered_df = df[df["cyclist_count"] > 0]
  findings = []

  for idx, row in filtered_df.iterrows():
    findings.append({
    "check_id": "C2",
    "segment_id": row["segment_id"],
    "severity": "info",
    "description": f"Scene contains {row['cyclist_count']} cyclist(s) - vulnerable road user present."
    })

  return findings

def check_empty_scene(df):
  filtered_df = df[df["vehicle_count"] == 0]
  findings = []
  for idx, row in filtered_df.iterrows():
    findings.append({
    "check_id": "C3",
    "segment_id": row["segment_id"],
    "severity": "warning",
    "description": "Scene contains 0 vehicles - possible data issue or unusual location."
    })

  return findings


def check_no_night_coverage(df):
  #function checks for non-day data
  findings = []
  non_day_df = df[df["time_of_day"] != "Day"]
  non_day_count = len(non_day_df)

  if non_day_count == 0:
    findings.append({
        "check_id": "C4",
        "segment_id": "ALL",
        "severity": "warning",
        "description": f"0 of {len(df)} segments are non_daytime. No night coverage in sample."
    })
    return findings

  else:
    return []



def check_no_adverse_weather_coverage(df):
  #function checks for non-sunny days
  findings = []
  non_adverse_weather_df = df[df["weather"] != "sunny"]
  non_adverse_weather_count = len(non_adverse_weather_df)

  if non_adverse_weather_count == 0:
    findings.append({
        "check_id": "C5",
        "segment_id": "ALL",
        "severity": "warning",
        "description": f"0 of {len(df)} segments are non_adverse_weather. No adverse weather conditions coverage in sample."
    })
    return findings

  else:
    return []




master_list_of_checks = []
all_checks = [check_high_pedestrian_density,
              check_vulnerable_road_user,
              check_empty_scene,
              check_no_night_coverage,
              check_no_adverse_weather_coverage
              ]

for check in all_checks:
  master_list_of_checks.extend(check(segment_df))



segment_checks_df = pd.DataFrame(master_list_of_checks)
segment_checks_df.to_csv('/content/findings.csv', index=False)
print("CSV file successfully saved in your Colab folder!")


CSV file successfully saved in your Colab folder!
